**Release-candidate copy.** Sanitized (private paths / account handle redacted; see REDACTION_MANIFEST.md). A guard cell was prepended so nothing runs, downloads, or calls a paid API by accident. Executed outputs (if any) are the original run outputs.

In [ ]:
# RELEASE GUARD (added to the release copy; not part of the original run)
import os
if os.environ.get("WSP_ALLOW_NOTEBOOK_RUN") != "1":
    raise SystemExit("Saved results are in runs/. Re-running this notebook trains models and downloads data/models (GPU, hours). Set WSP_ALLOW_NOTEBOOK_RUN=1 to proceed.")
if os.environ.get("WSP_PAID_API_OPT_IN") != "1":
    # Paid Anthropic calls stay OFF unless explicitly opted in; cache/dry-run only.
    os.environ.update({"WSP_RUN_CLAUDE": "0", "WSP_DRY_RUN_CLAUDE": "1", "WSP_RUN_C": "0"})
    os.environ.pop("ANTHROPIC_API_KEY", None)


# WSP v3: Gradient Descent vs Weight-Space Programming (harness-fixed, budget-safe)
Changes from v2: full safe-builtin sandbox (incl. `next`, whitelisted imports, audit hook, separate process), robust code extraction
(fences/prose/truncation, `stop_reason` logged), honest model labels (no silent fallback), normalised margins, accuracy-only CIFAR,
recall-resistant task `permadd`, deterministic/seeded local sampling, and a **Claude spend guard**:
`RUN_CLAUDE=False` and `DRY_RUN_CLAUDE=True` by default, $3.00 total cap, per-section caps, pre-run estimate, reply cache.
**Run 1 (recommended):** leave Claude off -> A baselines + oracles + local models. **Run 2:** set `RUN_CLAUDE=True, DRY_RUN_CLAUDE=False`.

In [ ]:
# ---------------- Install (Kaggle: Internet ON) ----------------
# Only packages that Kaggle's image may lack/outdate. torch/torchvision/numpy/pandas are preinstalled; never reinstall torch.
import os, subprocess, sys
ON_KAGGLE = os.path.isdir("/kaggle/working")
if ON_KAGGLE:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "anthropic>=0.40", "bitsandbytes>=0.43", "accelerate>=0.33"],
                       capture_output=True, text=True)
    print("pip rc", r.returncode, r.stderr[-800:])

## Configuration (edit the Claude spend controls here)

In [ ]:
# ---------------- Configuration ----------------
import os, json, time, math, random, platform, re, traceback, gc, tempfile, subprocess, sys, hashlib
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import pandas as pd
import matplotlib; import matplotlib.pyplot as plt
ON_KAGGLE = os.path.isdir("/kaggle/working")
_env = lambda k, d: os.environ.get(k, d)
QUICK = _env("WSP_QUICK", "0") == "1"   # tiny CPU smoke test
MOCK  = _env("WSP_MOCK", "0") == "1"    # replace local LLMs by a mock synthesizer (no downloads)

RUN_TASKS = {"modadd": True, "modmul": True, "parity": True, "permadd": True, "cifar": True}
RUN_A = True
RUN_LOCAL = _env("WSP_RUN_LOCAL", "1") == "1"      # local open-weight synthesizers
RUN_ORACLE_CHECK = True

# ======== CLAUDE SPEND CONTROLS (read carefully) ========
RUN_CLAUDE     = _env("WSP_RUN_CLAUDE", "0") == "1"   # master switch. False -> no Claude section at all.
DRY_RUN_CLAUDE = _env("WSP_DRY_RUN_CLAUDE", "1") == "1"  # True -> NEVER construct the Anthropic client; use cache / v2 replay / mock.
CLAUDE_BUDGET_USD = float(_env("CLAUDE_BUDGET_USD", "3.00"))    # HARD total cap for this run
CLAUDE_SECTION_CAP = {"B_algo": 2.00, "B_cifar": 0.80, "C": 0.30}   # HARD per-section caps
RUN_C_CLAUDE = _env("WSP_RUN_C", "0") == "1"                              # agentic editing with Claude (off by default)
# Cases to (re)run with Claude: (task, prompt_level, list_of_rep_indices). Reps 0-9 of v2 are already in the cache/replay,
# so fresh samples start at rep 10. Each (model, prompt, rep) is cached: a re-run never pays twice.
RUN_CLAUDE_ONLY_CASES = [("modmul", "full", list(range(10, 15))), ("modmul", "partial", list(range(10, 15))),
                         ("modmul", "none", list(range(10, 15))), ("cifar", "full", [10, 11]), ("modadd", "none", [])]
CLAUDE_MODEL_ID    = _env("CLAUDE_MODEL_ID", "claude-sonnet-5-5")
CLAUDE_TEMPERATURE = 0.0   # retried without temperature if the model rejects it
CLAUDE_MAX_TOKENS  = {"algo": 6000, "cifar": 12000, "C": 6000}
PRICE_PER_MTOK     = {"input": 2.0, "output": 10.0}
CACHE_DIR = _env("WSP_CACHE_DIR", "/kaggle/working/llm_cache" if ON_KAGGLE else "./llm_cache")
# Optional: v2 llm_calls.jsonl (e.g. attached as a Kaggle dataset) to seed the cache and replay offline.
V2_LLM_LOG = _env("WSP_V2_LLM_LOG", next((p for p in ["/kaggle/input/wsp-v2-outputs/llm_calls.jsonl", "./v2_outputs/llm_calls.jsonl"]
                                          if os.path.exists(p)), ""))

# Local models (one loaded at a time). Rows are labelled by the model id that ACTUALLY loaded.
SYNTHESIZERS = [
    {"name": "qwen2.5-coder-14b", "kind": "hf", "model_id": "Qwen/Qwen2.5-Coder-14B-Instruct"},
    {"name": "qwen2.5-7b", "kind": "hf", "model_id": "Qwen/Qwen2.5-7B-Instruct"},
    {"name": "llama-3.1-8b", "kind": "hf", "model_id": "meta-llama/Llama-3.1-8B-Instruct", "requires_hf_token": True},
]
if MOCK: SYNTHESIZERS = [{"name": "mock", "kind": "mock", "model_id": "mock"}]
LOCAL_MAX_NEW_TOKENS = 2048
LOCAL_DETERMINISTIC = False          # True -> greedy (do_sample=False): repeats collapse to 1 per cell
LOCAL_TEMPERATURE, LOCAL_TOP_P, LOCAL_SEED_BASE = 0.7, 0.95, 1000   # sampling is seeded per repeat (seed = base + rep)

PROMPT_LEVELS = ["full", "partial", "none"]
N_REPEATS_LOCAL = 3
N_REPEATS_CIFAR_LOCAL = 2
SUCCESS_THRESH = 0.99      # algorithmic tasks only. CIFAR reports accuracy only.
CIFAR_CHANCE, CIFAR_ABOVE_CHANCE_PTS = 0.10, 0.05

SEEDS_A = [0, 1, 2]
LR_GRID = {"adamw": [3e-4, 1e-3, 3e-3], "sgd": [0.03, 0.1, 0.3]}
WEIGHT_DECAY = {"adamw": 1.0, "sgd": 1e-4}
CHECKPOINTS = [100, 500, 1000, 3000, 10000, 20000]
SPLIT = (0.3, 0.1)
CIFAR_STEPS = [200, 1000, 3000]
CIFAR_LR_GRID = {"adam": [3e-4, 1e-3], "sgd": [0.01, 0.05]}
CIFAR_BATCH = 128
C_MAX_TARGETS_PER_TASK, C_ITERS = 2, 2
TIME_LIMIT = {"A_algo": 75*60, "A_cifar": 25*60, "B_claude": 60*60, "B_hf_each": 70*60, "C": 20*60}
SANDBOX_TIMEOUT = {"algo": 120, "cifar": 300}

if QUICK:
    SEEDS_A = [0]; LR_GRID = {"adamw": [1e-3], "sgd": [0.1]}; CHECKPOINTS = [20, 50]
    CIFAR_STEPS = [5, 10]; CIFAR_LR_GRID = {"adam": [1e-3]}
    N_REPEATS_LOCAL = 2; N_REPEATS_CIFAR_LOCAL = 1; C_ITERS = 1

OUT_DIR = "/kaggle/working" if ON_KAGGLE else os.path.abspath(_env("WSP_OUT", "./wsp_outputs_v3"))
GEN_DIR = f"{OUT_DIR}/generated_code"; os.makedirs(GEN_DIR, exist_ok=True); os.makedirs(CACHE_DIR, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
T_START = time.time()
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
print("device", DEVICE, "| torch", torch.__version__, "| out", OUT_DIR, "| QUICK", QUICK, "| MOCK", MOCK,
      "| RUN_CLAUDE", RUN_CLAUDE, "| DRY_RUN_CLAUDE", DRY_RUN_CLAUDE, "| budget $%.2f" % CLAUDE_BUDGET_USD)

## Tasks, architectures, oracles, prompts (modadd/modmul/parity prompts identical to v2)

In [ ]:
P = 113
H_ALGO = 256
PAR_N, PAR_K, H_PAR = 16, 6, 64
PERM = np.random.default_rng(4321).permutation(113)   # secret relabelling for the recall-resistant task
PAR_SUBSET = sorted(np.random.default_rng(1234).choice(PAR_N, PAR_K, replace=False).tolist())

def primitive_root(p):
    fac = [q for q in range(2, p) if (p - 1) % q == 0 and all(q % r for r in range(2, int(q**.5) + 1))]
    return next(g for g in range(2, p) if all(pow(g, (p - 1) // q, p) != 1 for q in fac))
G_ROOT = primitive_root(P)

class MLP(nn.Module):
    """Shared architecture for A/B/C. Params: W1 (H,in), [b1 (H)], W2 (out,H), [b2 (out)]."""
    def __init__(self, d_in, H, d_out, act="quad", bias=False):
        super().__init__()
        self.act_name, self.bias = act, bias
        self.W1 = nn.Parameter(torch.randn(H, d_in) / math.sqrt(d_in))
        self.W2 = nn.Parameter(torch.randn(d_out, H) / math.sqrt(H))
        if bias: self.b1 = nn.Parameter(torch.zeros(H)); self.b2 = nn.Parameter(torch.zeros(d_out))
    def forward(self, x):
        h = x @ self.W1.T + (self.b1 if self.bias else 0)
        h = h * h if self.act_name == "quad" else F.relu(h)
        return h @ self.W2.T + (self.b2 if self.bias else 0)

def onehot2(a, b, p=P): return torch.cat([F.one_hot(a, p), F.one_hot(b, p)], -1).float()

def make_task(name):
    if name == "permadd":
        a, b = np.meshgrid(np.arange(P), np.arange(P), indexing="ij"); a, b = a.ravel(), b.ravel()
        inv = np.argsort(PERM); y = inv[(PERM[a] + PERM[b]) % P]
        X = onehot2(torch.tensor(a), torch.tensor(b)); Y = torch.tensor(y)
        shapes = {"W1": (H_ALGO, 2 * P), "W2": (P, H_ALGO)}
        return dict(name=name, X=X, Y=Y, shapes=shapes, n_cls=P, make_model=lambda: MLP(2 * P, H_ALGO, P, "quad", False))
    if name in ("modadd", "modmul"):
        lo = 0 if name == "modadd" else 1
        a, b = np.meshgrid(np.arange(lo, P), np.arange(lo, P), indexing="ij"); a, b = a.ravel(), b.ravel()
        y = (a + b) % P if name == "modadd" else (a * b) % P
        X = onehot2(torch.tensor(a), torch.tensor(b)); Y = torch.tensor(y)
        shapes = {"W1": (H_ALGO, 2 * P), "W2": (P, H_ALGO)}
        return dict(name=name, X=X, Y=Y, shapes=shapes, n_cls=P,
                    make_model=lambda: MLP(2 * P, H_ALGO, P, "quad", False))
    if name == "parity":
        idx = np.arange(2 ** PAR_N); bits = ((idx[:, None] >> np.arange(PAR_N)) & 1).astype(np.float32)
        y = bits[:, PAR_SUBSET].sum(1).astype(int) % 2
        shapes = {"W1": (H_PAR, PAR_N), "b1": (H_PAR,), "W2": (2, H_PAR), "b2": (2,)}
        return dict(name=name, X=torch.tensor(bits), Y=torch.tensor(y), shapes=shapes, n_cls=2,
                    make_model=lambda: MLP(PAR_N, H_PAR, 2, "relu", True))
    raise ValueError(name)

def splits(n, seed):
    perm = np.random.default_rng(seed).permutation(n); a, b = int(SPLIT[0] * n), int((SPLIT[0] + SPLIT[1]) * n)
    return torch.tensor(perm[:a]), torch.tensor(perm[a:b]), torch.tensor(perm[b:])

# ---- Oracle constructors, kept as SOURCE strings so the mock synthesizer can return them as "LLM code" ----
ORACLE_SRC = {}
ORACLE_SRC["modadd"] = r"""
def build_weights():
    p, H, M = 113, 256, 3
    K = H // (M*M); W1 = np.zeros((H, 2*p)); W2 = np.zeros((p, H)); x = np.arange(p); h = 0
    for k in range(1, K+1):
        w = 2*np.pi*k/p
        for i in range(M):
            for j in range(M):
                phi, psi = 2*np.pi*i/M, 2*np.pi*j/M
                W1[h, :p] = np.cos(w*x+phi); W1[h, p:] = np.cos(w*x+psi); W2[:, h] = np.cos(w*x+phi+psi); h += 1
    return {"W1": W1, "W2": W2 / (K*M*M/2)}
"""
ORACLE_SRC["modmul"] = r"""
def build_weights():
    p, H, M, g = 113, 256, 3, GROOT
    n = p - 1; log = np.zeros(p, dtype=int); v = 1
    for e in range(n): log[v] = e; v = (v*g) % p
    K = H // (M*M); W1 = np.zeros((H, 2*p)); W2 = np.zeros((p, H)); L = log[1:]; h = 0
    for k in range(1, K+1):
        w = 2*np.pi*k/n
        for i in range(M):
            for j in range(M):
                phi, psi = 2*np.pi*i/M, 2*np.pi*j/M
                W1[h, 1:p] = np.cos(w*L+phi); W1[h, p+1:] = np.cos(w*L+psi); W2[1:, h] = np.cos(w*L+phi+psi); h += 1
    W2[0, :] = 0.0
    return {"W1": W1, "W2": W2 / (K*M*M/2)}
""".replace("GROOT", str(G_ROOT))
ORACLE_SRC["permadd"] = r"""
def build_weights():
    p, H, M = 113, 256, 3
    V = np.array(PERMLIST)
    K = H // (M*M); W1 = np.zeros((H, 2*p)); W2 = np.zeros((p, H)); h = 0
    for k in range(1, K+1):
        w = 2*np.pi*k/p
        for i in range(M):
            for j in range(M):
                phi, psi = 2*np.pi*i/M, 2*np.pi*j/M
                W1[h, :p] = np.cos(w*V+phi); W1[h, p:] = np.cos(w*V+psi); W2[:, h] = np.cos(w*V+phi+psi); h += 1
    return {"W1": W1, "W2": W2 / (K*M*M/2)}
""".replace("PERMLIST", str(PERM.tolist()))
ORACLE_SRC["parity"] = r"""
def build_weights():
    n, H, S = 16, 64, SUBSET
    k = len(S); W1 = np.zeros((H, n)); b1 = np.zeros(H); W2 = np.zeros((2, H)); b2 = np.array([1.0, -1.0])
    for j in range(k+1):
        W1[j, S] = 1.0; b1[j] = -j
        c = 2.0 if j == 0 else 4.0*(-1)**j
        W2[1, j] = c; W2[0, j] = -c
    return {"W1": W1, "b1": b1, "W2": W2, "b2": b2}
""".replace("SUBSET", str(PAR_SUBSET))

SYSTEM_PROMPT = ("You are an expert in mechanistic interpretability who writes neural network weights directly, "
                 "without any training, data fitting or gradients.")
FORMAT_RULES = """Return ONLY one python code block defining `def build_weights():` (no arguments) that uses only `np` (numpy, already
imported) and `math`; no other imports, no file or network access, no randomness unless seeded. It must return a dict mapping
each parameter name to a numpy array of exactly the shape listed. Do not print large arrays; compute them in code."""

def task_prompt(task, level):
    """Prompts for modadd/modmul/parity are byte-identical to v2 so v2 replies are valid cache entries."""
    if task == "modadd":
        spec = f"""Construct weights for a network computing (a+b) mod p, p={P}, for ALL p^2 pairs a,b in 0..{P-1}.
Architecture (fixed): x = concat(onehot(a), onehot(b)) (dim {2*P}); h = W1 @ x, W1 shape ({H_ALGO}, {2*P}), no bias;
act = h*h (elementwise square); logits = W2 @ act, W2 shape ({P}, {H_ALGO}), no bias. argmax(logits) must equal (a+b) mod p."""
        hint = {"full": """Hint: Fourier features. With w_k = 2*pi*k/p, embed a and b as cos(w_k a + phase), cos(w_k b + phase'); squaring
yields cross terms cos(w_k(a+b)+phase+phase') via cos x cos y = [cos(x+y)+cos(x-y)]/2. Read out with W2[c,h] = cos(w_k c + phase+phase')
so logit_c ~ sum_k cos(w_k(a+b-c)). Use a grid of phases so unwanted terms (cos 2wa, cos 2wb, cos w(a-b), constants) cancel.""",
                "partial": "Hint: a solution exists using periodic (trigonometric) features and trig product identities.",
                "none": ""}[level]
    elif task == "modmul":
        spec = f"""Construct weights for a network computing (a*b) mod p, p={P}, for ALL pairs a,b in 1..{P-1} (0 never appears as input).
Architecture (fixed): x = concat(onehot(a), onehot(b)) with one-hot over 0..{P-1} (dim {2*P}); h = W1 @ x, W1 shape ({H_ALGO}, {2*P}), no bias;
act = h*h; logits = W2 @ act, W2 shape ({P}, {H_ALGO}), no bias, logits indexed by c in 0..{P-1}. argmax(logits) must equal (a*b) mod p."""
        hint = {"full": f"""Hint: the multiplicative group mod p is cyclic of order {P-1} with generator g={G_ROOT}. Let log(a) be the discrete log base g.
Then a*b mod p corresponds to log(a)+log(b) mod {P-1}. Apply the Fourier construction for modular addition on Z_{P-1}: embed
cos(2*pi*k*log(a)/{P-1} + phase), square, read out with cos(2*pi*k*log(c)/{P-1} + phase+phase'), phase grid so unwanted terms cancel; row c=0 can be zero.""",
                "partial": "Hint: use the structure of the multiplicative group mod p (it is cyclic) and periodic features.",
                "none": ""}[level]
    elif task == "permadd":
        spec = f"""Symbols 0..{P-1} encode the elements of Z_{P} through a SECRET relabelling: symbol s stands for the integer VAL[s], where
VAL = {PERM.tolist()}.
Construct weights for a network that, given symbols a and b, outputs the symbol c with VAL[c] = (VAL[a] + VAL[b]) mod {P}, for ALL {P}^2 pairs.
Architecture (fixed): x = concat(onehot(a), onehot(b)) over symbols (dim {2*P}); h = W1 @ x, W1 shape ({H_ALGO}, {2*P}), no bias;
act = h*h (elementwise square); logits = W2 @ act, W2 shape ({P}, {H_ALGO}), no bias, indexed by output symbol c."""
        hint = {"full": """Hint: Fourier features on the relabelled values. With w_k = 2*pi*k/p, embed symbol a as cos(w_k VAL[a] + phase), likewise b;
squaring yields cos(w_k(VAL[a]+VAL[b]) + phase+phase'). Read out with W2[c,h] = cos(w_k VAL[c] + phase+phase'); use a phase grid so unwanted terms cancel.""",
                "partial": "Hint: the relabelled operation is a cyclic group; periodic features of the underlying values work.",
                "none": ""}[level]
    elif task == "parity":
        spec = f"""Construct weights for a network computing the parity (XOR) of the input bits at positions S={PAR_SUBSET} (0-indexed) of a
{PAR_N}-bit input x in {{0,1}}^{PAR_N}, for ALL 2^{PAR_N} inputs; the other bits are distractors.
Architecture (fixed): h = relu(W1 @ x + b1), W1 ({H_PAR}, {PAR_N}), b1 ({H_PAR},); logits = W2 @ h + b2, W2 (2, {H_PAR}), b2 (2,).
Class 1 = odd parity, class 0 = even. argmax(logits) must be correct for every input."""
        hint = {"full": f"""Hint: let s = sum of the bits in S (an integer 0..{PAR_K}). Use units relu(s - j) for j=0..{PAR_K} and choose output weights
so the piecewise-linear function of s equals +1 at odd s and -1 at even s (a zig-zag); logits = [-f, f] plus bias.""",
                "partial": "Hint: the answer only depends on the count of active bits in S; a piecewise-linear function of that count suffices.",
                "none": ""}[level]
    shapes = TASKS[task]["shapes"]
    return f"{spec}\n{hint}\nParameter names and shapes: {json.dumps({k: list(v) for k, v in shapes.items()})}\n{FORMAT_RULES}"

TASKS = {t: make_task(t) for t in ["modadd", "modmul", "parity", "permadd"] if RUN_TASKS.get(t)}
for t, T in TASKS.items(): print(t, "N =", len(T["Y"]), "shapes", T["shapes"])
print("primitive root mod", P, "=", G_ROOT, "| parity subset", PAR_SUBSET)

## Sandbox, extraction, evaluation, metrics; oracle sanity check

In [ ]:
# ---------------- Sandboxed execution, robust code extraction, evaluation, metrics ----------------
# Fault-isolation harness (separate process + timeout + whitelist imports + audit hook). It is NOT a hardened security sandbox;
# all generated code is logged for review.
RUNNER = r'''
import sys, json, math, builtins as _b
import numpy as np
src_path, out_path, fn_name, in_path, want_torch = sys.argv[1], sys.argv[2], sys.argv[3], sys.argv[4], sys.argv[5] == "1"
src = open(src_path).read()
INPUTS = dict(np.load(in_path))
import itertools, functools, collections, operator, cmath, fractions, random, typing, string, statistics, numbers, dataclasses, copy, heapq, bisect
import numpy.linalg, numpy.fft, numpy.random, numpy.polynomial, numpy.lib.stride_tricks
if want_torch:
    import torch, torch.nn, torch.nn.functional
ALLOWED_MODULES = {"numpy", "math", "itertools", "functools", "collections", "operator", "cmath", "fractions", "random", "typing",
                   "string", "statistics", "numbers", "dataclasses", "copy", "heapq", "bisect", "torch", "scipy"}
_real_import = _b.__import__
def _safe_import(name, globals=None, locals=None, fromlist=(), level=0):
    root = name.split(".")[0]
    if level != 0 or root not in ALLOWED_MODULES:
        raise ImportError(f"import of {name!r} is blocked in the sandbox")
    if root == "torch" and "torch" not in sys.modules:
        raise ImportError("torch not preloaded (pass want_torch)")
    return _real_import(name, globals, locals, fromlist, level)
SAFE_NAMES = ["abs","all","any","ascii","bin","bool","bytearray","bytes","callable","chr","complex","dict","divmod","enumerate",
  "filter","float","format","frozenset","hasattr","hash","hex","id","int","isinstance","issubclass","iter","len","list","map","max",
  "min","next","object","oct","ord","pow","print","range","repr","reversed","round","set","slice","sorted","str","sum","super",
  "tuple","type","zip","getattr","setattr","property","staticmethod","classmethod","NotImplemented","Ellipsis","None","True","False",
  "Exception","ValueError","TypeError","KeyError","IndexError","AssertionError","ArithmeticError","ZeroDivisionError",
  "RuntimeError","StopIteration","NotImplementedError","AttributeError","OverflowError","LookupError","ImportError","__build_class__"]
safe = {k: getattr(_b, k) for k in SAFE_NAMES if hasattr(_b, k)}
safe["__import__"] = _safe_import
def _blocked(name):
    def f(*a, **k): raise PermissionError(f"sandbox: builtin {name}() is blocked")
    return f
for _n in ["open", "eval", "exec", "compile", "input", "breakpoint", "globals", "locals", "vars", "memoryview", "exit", "quit", "help"]:
    safe[_n] = _blocked(_n)
_ALLOW = [False]
BLOCK_PREFIX = ("subprocess.", "os.system", "os.exec", "os.spawn", "os.fork", "os.posix_spawn", "os.kill", "os.remove", "os.rmdir",
                "os.rename", "os.unlink", "os.truncate", "os.chmod", "os.chown", "shutil.", "socket.", "urllib.", "http.", "ctypes.",
                "pty.", "fcntl.", "os.putenv", "os.chdir", "code.__new__")
def _hook(ev, args):
    if _ALLOW[0]: return
    if ev == "open":
        path, mode = str(args[0]), str(args[1] or "r")
        if path.endswith((".py", ".pyc", ".so")) and path.startswith(_LIB_ROOTS) and not any(c in mode for c in "wax+"):
            return                                   # lazy imports of stdlib / site-packages modules (read-only)
        raise PermissionError(f"sandbox: file access blocked ({args[0]!r})")
    if ev.startswith(BLOCK_PREFIX):
        raise PermissionError(f"sandbox: blocked operation {ev}")
_STAGE = [0]
import sysconfig as _sc
_LIB_ROOTS = tuple({sys.prefix, sys.base_prefix, sys.exec_prefix, _sc.get_paths()["stdlib"], _sc.get_paths()["purelib"], _sc.get_paths()["platlib"]})
ns = {"np": np, "numpy": np, "math": math, "__builtins__": safe, "__name__": "generated"}
if want_torch: ns.update(torch=torch, nn=torch.nn, F=torch.nn.functional)
code_obj = compile(src, "<generated>", "exec")
sys.addaudithook(_hook)
exec(code_obj, ns)           # audit 'exec' allowed only for this top-level call
_STAGE[0] = 1
if fn_name not in ns or not callable(ns[fn_name]):
    names = [k for k, v in ns.items() if callable(v) and not k.startswith("_") and k not in ("np", "numpy", "math", "torch", "nn", "F")]
    print("SANDBOX_MISSING_FN " + json.dumps(names), file=sys.stderr); sys.exit(3)
W = ns[fn_name]() if fn_name == "build_weights" else ns[fn_name](dict(INPUTS))
def _arr(v):
    if hasattr(v, "detach"): v = v.detach().cpu().numpy()
    return np.asarray(v, dtype=np.float64)
if not isinstance(W, dict):
    print("SANDBOX_NOT_DICT " + type(W).__name__, file=sys.stderr); sys.exit(4)
arrs = {str(k): _arr(v) for k, v in W.items()}
_ALLOW[0] = True
np.savez(out_path, **arrs)
'''

class SandboxError(RuntimeError):
    def __init__(self, kind, msg): super().__init__(f"[{kind}] {msg}"); self.kind = kind

FENCE_RE = re.compile(r"(?:^|\n)[ \t]*(```|~~~)[ \t]*([A-Za-z0-9_+-]*)[^\n]*\n(.*?)(?:\n[ \t]*\1[ \t]*(?=\n|$)|\Z)", re.S)

def _compiles(s):
    try: compile(s, "<x>", "exec"); return True
    except SyntaxError: return False

def extract_code(text, fn_name="build_weights", stop_reason=None):
    """Return (code, info). Handles ```python/```py/``` fences, multiple fences, leading/trailing prose, unclosed (truncated) fences.
    Never strips imports. info = {method, n_fences, truncated, compiles}."""
    text = (text or "").replace("\r\n", "\n")
    info = {"n_fences": 0, "truncated": stop_reason == "max_tokens", "method": None, "compiles": False}
    blocks = []
    for m in FENCE_RE.finditer(text):
        lang, body = m.group(2).lower(), m.group(3)
        closed = bool(re.search(r"\n[ \t]*" + re.escape(m.group(1)) + r"[ \t]*$", m.group(0)))
        if lang in ("", "python", "py", "python3", "numpy"): blocks.append((body, closed))
    info["n_fences"] = len(blocks)
    if any(not c for _, c in blocks): info["truncated"] = True
    cands = []
    if blocks:
        with_fn = [b for b, _ in blocks if re.search(rf"def\s+{fn_name}\s*\(", b)]
        # several blocks: concatenate (helpers may live in other blocks; a later re-definition wins, i.e. the final version)
        if len(blocks) > 1: cands.append(("fences_concat", "\n\n".join(b for b, _ in blocks)))
        if with_fn: cands.append(("fence_with_fn", with_fn[-1]))
        cands.append(("longest_fence", max((b for b, _ in blocks), key=len)))
    else:
        lines = text.split("\n")
        start = next((i for i, l in enumerate(lines) if re.match(r"\s*(import |from \w|def |@|[A-Za-z_]\w*\s*=)", l)), None)
        if start is not None:
            body = lines[start:]
            for end in range(len(body), 0, -1):          # trim trailing prose until it compiles
                s = "\n".join(body[:end])
                if _compiles(s): cands.append(("unfenced", s)); break
            else: cands.append(("unfenced_raw", "\n".join(body)))
        else: cands.append(("raw", text))
    for method, s in cands:
        if _compiles(s) and re.search(rf"def\s+{fn_name}\s*\(", s):
            info.update(method=method, compiles=True); return s, info
    method, s = cands[0]
    info.update(method=method, compiles=_compiles(s)); return s, info

def classify_error(msg):
    m = str(msg)
    for kind, pats in [("truncated", ["[truncated]"]), ("timeout", ["[timeout]", "TimeoutExpired"]),
                       ("blocked", ["sandbox: ", "is blocked in the sandbox"]), ("syntax", ["SyntaxError", "IndentationError"]),
                       ("missing_fn", ["SANDBOX_MISSING_FN", "[missing_fn]"]), ("not_dict", ["SANDBOX_NOT_DICT", "weights must be a dict"]),
                       ("shape", ["shape", "missing parameter"]), ("nonfinite", ["non-finite"]), ("name_error", ["NameError"]),
                       ("import", ["ImportError", "ModuleNotFoundError"]), ("no_code", ["[no_code]"])]:
        if any(p in m for p in pats): return kind
    return "runtime"

def run_generated(code_str, fn_name="build_weights", inputs=None, timeout=120, want_torch=None):
    """Run generated code in a SUBPROCESS. Returns dict of float64 arrays. Raises SandboxError(kind, ...)."""
    if want_torch is None: want_torch = bool(re.search(r"\btorch\b", code_str))
    if not _compiles(code_str):
        try: compile(code_str, "<generated>", "exec")
        except SyntaxError as e: raise SandboxError("syntax", f"SyntaxError: {e}")
    d = tempfile.mkdtemp(); src, out, inp, run = f"{d}/g.py", f"{d}/w.npz", f"{d}/in.npz", f"{d}/runner.py"
    open(src, "w").write(code_str); open(run, "w").write(RUNNER)
    np.savez(inp, **(inputs or {"_": np.zeros(1)}))
    env = {k: v for k, v in os.environ.items() if k not in ("ANTHROPIC_API_KEY", "HF_TOKEN", "KAGGLE_KEY")}
    env.update(OMP_NUM_THREADS="4", CUDA_VISIBLE_DEVICES="")
    try:
        r = subprocess.run([sys.executable, "-I", run, src, out, fn_name, inp, "1" if want_torch else "0"],
                           capture_output=True, text=True, timeout=timeout, cwd=d, env=env)
    except subprocess.TimeoutExpired:
        raise SandboxError("timeout", f"generated code exceeded {timeout}s")
    if r.returncode != 0:
        err = r.stderr[-800:]
        raise SandboxError(classify_error(err), "generated code failed: " + err)
    return {k: v for k, v in np.load(out).items()}

def validate(W, shapes):
    if not isinstance(W, dict): raise SandboxError("not_dict", "weights must be a dict")
    out = {}
    for k, shp in shapes.items():
        if k not in W: raise SandboxError("shape", f"missing parameter {k} (got {sorted(W)[:10]})")
        a = np.asarray(W[k], dtype=np.float64)
        if a.shape != tuple(shp) and a.size == int(np.prod(shp)): a = a.reshape(shp)
        if a.shape != tuple(shp): raise SandboxError("shape", f"{k} shape {a.shape} != {tuple(shp)}")
        if not np.isfinite(a).all(): raise SandboxError("nonfinite", f"{k} has non-finite values")
        out[k] = a
    return out

def load_weights(model, W):
    sd = model.state_dict()
    with torch.no_grad():
        for k in sd: sd[k].copy_(torch.tensor(W[k], dtype=sd[k].dtype))
    return model.to(DEVICE)

def get_weights(model): return {k: v.detach().cpu().numpy().astype(np.float64) for k, v in model.state_dict().items()}

def margin_stats(lg, y):
    """Raw logit margin (true - best other), margin normalised by per-example logit std (scale-free), and softmax-probability margin."""
    lg = lg.double(); c = lg.gather(1, y[:, None]).squeeze(1); o = lg.clone(); o.scatter_(1, y[:, None], -float("inf"))
    raw = c - o.max(-1).values
    sd = lg.std(-1).clamp_min(1e-12); norm = raw / sd
    p = torch.softmax(lg, -1); pc = p.gather(1, y[:, None]).squeeze(1); po = p.clone(); po.scatter_(1, y[:, None], -1.0)
    return raw, norm, pc - po.max(-1).values

@torch.no_grad()
def evaluate(model, X, Y, idx=None, noise=0.0, seed=0, bs=16384):
    model.eval().to(DEVICE)
    if idx is not None: X, Y = X[idx], Y[idx]
    g = torch.Generator().manual_seed(seed); preds, raws, norms, pms, losses = [], [], [], [], []
    for i in range(0, len(Y), bs):
        x = X[i:i+bs]
        if noise > 0: x = x + noise * torch.randn(x.shape, generator=g)
        x, y = x.to(DEVICE), Y[i:i+bs].to(DEVICE)
        lg = torch.nan_to_num(model(x).float(), nan=0.0, posinf=1e30, neginf=-1e30)
        preds.append(lg.argmax(-1).cpu()); r, n, pm = margin_stats(lg, y)
        raws.append(r.cpu()); norms.append(n.cpu()); pms.append(pm.cpu())
        losses.append(F.cross_entropy(lg.double(), y, reduction="sum").item())
    pred, r, n, pm = torch.cat(preds), torch.cat(raws), torch.cat(norms), torch.cat(pms)
    wrong = (pred != Y.cpu()).nonzero().squeeze(1)
    return {"acc": (pred == Y.cpu()).float().mean().item(), "loss": sum(losses) / len(Y),
            "mean_margin_raw": r.mean().item(), "min_margin_raw": r.min().item(),
            "mean_margin_norm": n.mean().item(), "min_margin_norm": n.min().item(),
            "mean_prob_margin": pm.mean().item(), "min_prob_margin": pm.min().item(),
            "wrong": wrong[:2000].tolist(), "pred": pred}

def fourier_energy(task, W):
    if task not in ("modadd", "modmul", "permadd") or "W1" not in W: return float("nan")
    E = W["W1"][:, :P]
    if task == "modmul": E = E[:, [pow(G_ROOT, e, P) for e in range(P - 1)]]
    if task == "permadd": E = E[:, np.argsort(PERM)]      # columns ordered by underlying value
    pw = (np.abs(np.fft.rfft(E, axis=1)) ** 2).sum(0)[1:]
    return float(np.sort(pw)[::-1][:10].sum() / (pw.sum() + 1e-12))

def structure(task, W):
    out = {}
    for name, A in W.items():
        if A.ndim < 2: continue
        A2 = A.reshape(A.shape[0], -1); s = np.linalg.svd(A2, compute_uv=False)
        if s[0] <= 0: out.update({f"{name}_rank": 0, f"{name}_eff_rank": 0.0, f"{name}_sparsity": 1.0}); continue
        q = s / s.sum()
        out[f"{name}_rank"] = int((s > 1e-3 * s[0]).sum()); out[f"{name}_eff_rank"] = float(np.exp(-(q @ np.log(q + 1e-12))))
        out[f"{name}_sparsity"] = float((np.abs(A2) < 1e-3 * np.abs(A2).max()).mean()); out[f"{name}_fro"] = float(np.linalg.norm(A2))
    out["fourier_top10_energy"] = fourier_energy(task, W)
    return out

NOISES = [0.1, 0.3, 0.5]
MARGIN_KEYS = ["mean_margin_raw", "min_margin_raw", "mean_margin_norm", "min_margin_norm", "mean_prob_margin", "min_prob_margin"]
def full_metrics(task, model, T, sp=None):
    r_all = evaluate(model, T["X"], T["Y"])
    row = {"all_acc": r_all["acc"], **{k: r_all[k] for k in MARGIN_KEYS}}
    if sp is not None: row["test_acc"] = evaluate(model, T["X"], T["Y"], sp[2])["acc"]
    for nz in NOISES: row[f"acc_noise{nz}"] = evaluate(model, T["X"], T["Y"], noise=nz)["acc"]
    row.update(structure(task, get_weights(model))); return row

def t_ci(x):
    x = np.asarray([v for v in x if v is not None and np.isfinite(v)], float); n = len(x)
    if n == 0: return np.nan, np.nan, np.nan
    if n == 1: return x[0], 0.0, 0.0
    try:
        from scipy import stats; tq = stats.t.ppf(0.975, n - 1)
    except Exception: tq = 1.96
    return x.mean(), x.std(ddof=1), tq * x.std(ddof=1) / math.sqrt(n)

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    ph = k / n; d = 1 + z * z / n; c = (ph + z * z / (2 * n)) / d; h = z * math.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))

PER_RUN, A_CURVES, C_ROWS = [], [], []
def save_all():
    pd.DataFrame(PER_RUN).drop(columns=["pred"], errors="ignore").to_csv(f"{OUT_DIR}/per_run.csv", index=False)
    pd.DataFrame(A_CURVES).to_csv(f"{OUT_DIR}/A_curves.csv", index=False)
    pd.DataFrame(C_ROWS).to_csv(f"{OUT_DIR}/C_edits.csv", index=False)
    json.dump({"per_run": PER_RUN, "C": C_ROWS, "llm_usage": globals().get("USAGE", {}), "elapsed_s": time.time() - T_START},
              open(f"{OUT_DIR}/results_partial.json", "w"), default=str)

ORACLE_W = {}
if RUN_ORACLE_CHECK:
    for t, T in TASKS.items():
        W = validate(run_generated(ORACLE_SRC[t]), T["shapes"]); ORACLE_W[t] = W
        m = load_weights(T["make_model"](), W); r = full_metrics(t, m, T)
        assert r["all_acc"] == 1.0, f"oracle {t} not exact: {r['all_acc']}"
        PER_RUN.append({"task": t, "instance": "B_oracle", "synth": "oracle", "synth_model": "analytic", "prompt": "-", "rep": 0, "ok": True, "success": True, **r})
        print(f"oracle {t}: acc {r['all_acc']:.4f} | norm min margin {r['min_margin_norm']:.3f} | prob margin {r['min_prob_margin']:.4f}")
    save_all()

## LLM layer: budget guard, cache, Claude, local models, mock

In [ ]:
# ---------------- LLM layer: budget guard, reply cache, Claude (dry-run by default), local HF models, mock ----------------
LLM_LOG = f"{OUT_DIR}/llm_calls.jsonl"
USAGE = {}
API_CALLS = {"claude": 0}          # number of REAL Anthropic API requests made in this process
class BudgetExceeded(Exception): pass

class Budget:
    def __init__(self, total, section_caps):
        self.total, self.caps, self.spent, self.by_section = float(total), dict(section_caps), 0.0, {}
    def check(self, section, worst):
        s = self.by_section.get(section, 0.0); cap = self.caps.get(section, self.total)
        if self.spent + worst > self.total: raise BudgetExceeded(f"total cap: spent ${self.spent:.3f} + worst ${worst:.3f} > ${self.total:.2f}")
        if s + worst > cap: raise BudgetExceeded(f"section {section} cap: spent ${s:.3f} + worst ${worst:.3f} > ${cap:.2f}")
    def add(self, section, cost):
        self.spent += cost; self.by_section[section] = self.by_section.get(section, 0.0) + cost
BUDGET = Budget(CLAUDE_BUDGET_USD, CLAUDE_SECTION_CAP)

def worst_case_cost(user, system, max_tokens):
    return ((len(user) + len(system)) / 3.0 * PRICE_PER_MTOK["input"] + max_tokens * PRICE_PER_MTOK["output"]) / 1e6

def _secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

def _log(rec):
    with open(LLM_LOG, "a") as f: f.write(json.dumps(rec, default=str) + "\n")
def _u(name):
    return USAGE.setdefault(name, {"calls": 0, "cached": 0, "input": 0, "output": 0, "cost_usd": 0.0, "latency_s": 0.0, "errors": 0})

# ---- reply cache keyed by (model, sha256(system+user), repeat idx) ----
def cache_key(model, system, user, rep):
    h = hashlib.sha256(json.dumps([system, user]).encode()).hexdigest()[:20]
    return f"{re.sub(r'[^A-Za-z0-9_.-]', '_', model)}__{h}__r{rep}"
def cache_get(model, system, user, rep, max_tokens):
    f = os.path.join(CACHE_DIR, cache_key(model, system, user, rep) + ".json")
    if not os.path.exists(f): return None
    c = json.load(open(f))
    if c.get("stop_reason") == "max_tokens" and c.get("max_tokens", 0) < max_tokens: return None   # truncated at a lower cap -> re-ask
    return c
def cache_put(model, system, user, rep, rec):
    f = os.path.join(CACHE_DIR, cache_key(model, system, user, rep) + ".json")
    json.dump(rec, open(f + ".tmp", "w")); os.replace(f + ".tmp", f)

def seed_cache_from_v2(path, max_tokens_v2=6000):
    """Import Claude replies saved by v2 (llm_calls.jsonl) into the cache. Tag format: B_<task>_<level>_r<rep> or B_cifar_r<rep>."""
    if not path or not os.path.exists(path): return 0
    n = 0
    for line in open(path):
        r = json.loads(line)
        if not str(r.get("synth", "")).startswith("claude") or "response" not in r or "user" not in r: continue
        m = re.match(r"B_(\w+?)_(?:(full|partial|none)_)?r(\d+)$", r.get("tag", ""))
        if not m: continue
        rep = int(m.group(3)); model = r.get("model_requested") or CLAUDE_MODEL_ID
        if cache_get(model, r["system"], r["user"], rep, 0) is None:
            cache_put(model, r["system"], r["user"], rep, {"response": r["response"], "stop_reason": r.get("stop_reason"),
                      "max_tokens": max_tokens_v2, "usage": r.get("usage"), "model_returned": r.get("model_returned"), "source": "v2_log", "tag": r["tag"]})
            n += 1
    return n

def _default_client_factory():
    key = _secret("ANTHROPIC_API_KEY")
    if not key: raise RuntimeError("No ANTHROPIC_API_KEY secret")
    import anthropic
    return anthropic.Anthropic(api_key=key)
CLIENT_FACTORY = _default_client_factory
_CLAUDE = {}

def call_claude(syn, system, user, tag, rep=0, section="B_algo", max_tokens=6000):
    model = syn["model_id"]; u = _u(syn["name"])
    c = cache_get(model, system, user, rep, max_tokens)
    if c is not None:
        u["cached"] += 1
        _log({"tag": tag, "synth": syn["name"], "event": "CACHE_HIT", "source": c.get("source"), "key": cache_key(model, system, user, rep)})
        return c["response"], {"in_tok": 0, "out_tok": 0, "cost_usd": 0.0, "latency_s": 0.0, "cached": True,
                               "stop_reason": c.get("stop_reason"), "reply_source": c.get("source", "cache")}
    if DRY_RUN_CLAUDE:            # never builds a client, never spends
        text, meta = call_mock(syn, system, user, tag)
        _log({"tag": tag, "synth": syn["name"], "event": "DRY_RUN_MOCK"})
        return text, {**meta, "cached": False, "stop_reason": "dry_run", "reply_source": "dry_run_mock"}
    worst = worst_case_cost(user, system, max_tokens)
    try: BUDGET.check(section, worst)
    except BudgetExceeded as e:
        _log({"tag": tag, "synth": syn["name"], "event": "BUDGET_STOP", "spent": BUDGET.spent, "why": str(e)}); raise
    if "client" not in _CLAUDE: _CLAUDE["client"] = CLIENT_FACTORY()
    client = _CLAUDE["client"]
    kw = dict(model=model, max_tokens=max_tokens, system=system, messages=[{"role": "user", "content": user}])
    t0 = time.perf_counter(); temp = CLAUDE_TEMPERATURE
    API_CALLS["claude"] += 1
    if _CLAUDE.get("no_temp"): temp = None; resp = client.messages.create(**kw)
    else:
        try: resp = client.messages.create(temperature=CLAUDE_TEMPERATURE, **kw)
        except Exception as e:
            if "temperature" not in str(e).lower(): raise
            _CLAUDE["no_temp"] = True; temp = None; API_CALLS["claude"] += 1; resp = client.messages.create(**kw)
    dt = time.perf_counter() - t0
    text = "".join(b.text for b in resp.content if getattr(b, "type", "") == "text")
    ui, uo = resp.usage.input_tokens, resp.usage.output_tokens
    cost = (ui * PRICE_PER_MTOK["input"] + uo * PRICE_PER_MTOK["output"]) / 1e6; BUDGET.add(section, cost)
    u["calls"] += 1; u["input"] += ui; u["output"] += uo; u["cost_usd"] += cost; u["latency_s"] += dt
    rec = {"response": text, "stop_reason": resp.stop_reason, "max_tokens": max_tokens, "usage": {"input": ui, "output": uo},
           "model_returned": resp.model, "source": "api", "tag": tag, "temperature": temp}
    cache_put(model, system, user, rep, rec)
    _log({"tag": tag, "synth": syn["name"], "model_requested": model, "model_returned": resp.model, "temperature": temp, "system": system,
          "user": user, "response": text, "usage": {"input": ui, "output": uo}, "cost_usd": cost, "cum_cost_usd": BUDGET.spent,
          "section": section, "latency_s": dt, "stop_reason": resp.stop_reason, "max_tokens": max_tokens, "time": time.strftime("%Y-%m-%dT%H:%M:%S%z")})
    if resp.stop_reason == "max_tokens": print(f"  WARNING {tag}: reply truncated at max_tokens={max_tokens}")
    return text, {"in_tok": ui, "out_tok": uo, "cost_usd": cost, "latency_s": dt, "cached": False, "stop_reason": resp.stop_reason, "reply_source": "api"}

# ---- local HF models ----
_HF = {}
def load_hf(syn):
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    tokn = _secret("HF_TOKEN"); tok_kw = {"token": tokn} if tokn else {}
    if syn.get("requires_hf_token") and not tokn:
        raise RuntimeError(f"SKIPPED {syn['model_id']}: gated model and no HF_TOKEN secret (no silent fallback in v3)")
    q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    tok = AutoTokenizer.from_pretrained(syn["model_id"], **tok_kw)
    mdl = AutoModelForCausalLM.from_pretrained(syn["model_id"], quantization_config=q, device_map="auto", torch_dtype=torch.float16, **tok_kw)
    mdl.eval(); _HF.update(tok=tok, model=mdl, id=syn["model_id"]); syn["resolved_id"] = syn["model_id"]

def unload_hf():
    _HF.clear(); gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

@torch.no_grad()
def call_hf(syn, system, user, tag, rep=0, **_):
    tok, mdl = _HF["tok"], _HF["model"]
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt")
    if not torch.is_tensor(ids): ids = ids["input_ids"]
    ids = ids.to(mdl.device); seed = LOCAL_SEED_BASE + rep; set_seed(seed)
    gen_kw = dict(max_new_tokens=LOCAL_MAX_NEW_TOKENS, pad_token_id=tok.eos_token_id, attention_mask=torch.ones_like(ids))
    gen_kw.update(do_sample=False) if LOCAL_DETERMINISTIC else gen_kw.update(do_sample=True, temperature=LOCAL_TEMPERATURE, top_p=LOCAL_TOP_P)
    t0 = time.perf_counter(); out = mdl.generate(ids, **gen_kw)
    dt = time.perf_counter() - t0; gen = out[0, ids.shape[1]:]; text = tok.decode(gen, skip_special_tokens=True)
    stop = "max_tokens" if len(gen) >= LOCAL_MAX_NEW_TOKENS else "end_turn"
    u = _u(syn["name"]); u["calls"] += 1; u["input"] += ids.shape[1]; u["output"] += len(gen); u["latency_s"] += dt
    _log({"tag": tag, "synth": syn["name"], "model_requested": syn["model_id"], "model_returned": _HF["id"], "seed": seed,
          "do_sample": not LOCAL_DETERMINISTIC, "temperature": None if LOCAL_DETERMINISTIC else LOCAL_TEMPERATURE, "system": system, "user": user,
          "response": text, "usage": {"input": int(ids.shape[1]), "output": int(len(gen))}, "stop_reason": stop, "latency_s": dt})
    return text, {"in_tok": int(ids.shape[1]), "out_tok": int(len(gen)), "cost_usd": 0.0, "latency_s": dt, "stop_reason": stop, "reply_source": "local"}

# ---- mock (offline) ----
_MOCK_RNG = np.random.default_rng(0)
def call_mock(syn, system, user, tag, **_):
    task = next((t for t in ORACLE_SRC if f"[[task={t}]]" in user), None)
    if task is None: task = next((t for t, k in [("permadd", "SECRET relabelling"), ("modmul", "(a*b) mod p"), ("modadd", "(a+b) mod p"),
                                                 ("parity", "parity (XOR)")] if k in user), None)
    r = _MOCK_RNG.random()
    if tag.startswith("C_"):
        text = "Here is the edit.\n```python\nimport numpy as np\ndef edit_weights(W):\n    W = dict(W)\n    W['W2'] = W['W2'] * 1.0\n    return W\n```"
    elif "CIFAR-10" in user:
        text = "```python\nimport numpy as np\ndef build_weights():\n    r = np.random.default_rng(0)\n    return {k: r.standard_normal(s)*0.1 for k, s in " + repr(CIFAR_SHAPES) + ".items()}\n```"
    elif task and r < 0.7:
        text = "Sure, here is the construction.\n```python\nimport numpy as np\nimport math\n" + ORACLE_SRC[task] + "```\nThis works because..."
    elif r < 0.85:
        text = "```python\ndef build_weights():\n    return {'W1': np.zeros((3,3))}\n```"
    else:
        text = "no code here"
    _u(syn["name"])["calls"] += 1
    _log({"tag": tag, "synth": syn["name"], "event": "MOCK", "response": text[:300]})
    return text, {"in_tok": 0, "out_tok": 0, "cost_usd": 0.0, "latency_s": 0.0, "stop_reason": "mock", "reply_source": "mock"}

def call_llm(syn, system, user, tag, rep=0, section="B_algo", max_tokens=6000):
    k = syn["kind"]
    try:
        if k == "claude": return call_claude(syn, system, user, tag, rep=rep, section=section, max_tokens=max_tokens)
        return {"hf": call_hf, "mock": call_mock}[k](syn, system, user, tag, rep=rep)
    except BudgetExceeded: raise
    except Exception:
        _u(syn["name"])["errors"] += 1; raise

N_SEEDED = seed_cache_from_v2(V2_LLM_LOG)
print("cache dir", CACHE_DIR, "| seeded from v2 log:", N_SEEDED, "| entries:", len(os.listdir(CACHE_DIR)))

## Instance A: gradient baselines

In [ ]:
def train_A_algo(task, T, opt_name, lr, seed, t_deadline):
    set_seed(seed); sp = splits(len(T["Y"]), seed); m = T["make_model"]().to(DEVICE)
    if opt_name == "adamw": opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=WEIGHT_DECAY["adamw"], betas=(0.9, 0.98))
    else: opt = torch.optim.SGD(m.parameters(), lr=lr, momentum=0.9, weight_decay=WEIGHT_DECAY["sgd"])
    xtr, ytr = T["X"][sp[0]].to(DEVICE), T["Y"][sp[0]].to(DEVICE); gpu_s = 0.0; rows = []
    for step in range(1, max(CHECKPOINTS) + 1):
        t0 = time.perf_counter(); m.train()
        loss = F.cross_entropy(m(xtr), ytr); opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        if DEVICE == "cuda": torch.cuda.synchronize()
        gpu_s += time.perf_counter() - t0
        if step in CHECKPOINTS:
            rows.append({"task": task, "opt": opt_name, "lr": lr, "seed": seed, "step": step, "train_loss": loss.item(),
                         "val_acc": evaluate(m, T["X"], T["Y"], sp[1])["acc"], "test_acc": evaluate(m, T["X"], T["Y"], sp[2])["acc"],
                         "train_compute_s": gpu_s})
        if time.time() > t_deadline: break
    return m, rows, sp

def run_A_algo():
    deadline = time.time() + TIME_LIMIT["A_algo"]
    for t, T in TASKS.items():
        for opt_name, lrs in LR_GRID.items():
            best = None
            for lr in lrs:
                for seed in SEEDS_A:
                    if time.time() > deadline: print("A_algo time limit hit"); return
                    try:
                        m, rows, sp = train_A_algo(t, T, opt_name, lr, seed, deadline)
                        A_CURVES.extend(rows); last = rows[-1]
                        r = full_metrics(t, m, T, sp)
                        PER_RUN.append({"task": t, "instance": "A", "synth": f"{opt_name}", "prompt": "-", "rep": seed, "lr": lr,
                                        "steps": last["step"], "val_acc": last["val_acc"], "compute_s": last["train_compute_s"], "ok": True, **r})
                        print(f"[A {t} {opt_name} lr={lr} s={seed}] val {last['val_acc']:.3f} test {r['test_acc']:.3f} ({last['train_compute_s']:.0f}s)")
                    except Exception as e:
                        print("A run failed:", t, opt_name, lr, seed, repr(e)); traceback.print_exc()
                    save_all()
if RUN_A and TASKS:
    try: run_A_algo()
    except Exception as e: print("A_algo section failed:", repr(e))

## Instance B helpers

In [ ]:
# ---------------- Instance B: one synthesis attempt -> extraction -> sandbox -> validation -> metrics ----------------
def label_of(syn):
    """Row label = model that actually produced the reply (never the requested-but-unavailable one)."""
    return syn.get("resolved_id") or syn["model_id"]

def process_reply(task, text, meta, fn_base, shapes=None, cifar=False):
    """Shared by live runs and offline replay. Returns row fields (ok, all_acc, error, error_kind, extraction info, metrics)."""
    row = {"ok": False, "all_acc": 0.0, "error": None, "error_kind": None}
    code_str, info = extract_code(text, "build_weights", meta.get("stop_reason"))
    row.update({f"extract_{k}": v for k, v in info.items()})
    open(f"{GEN_DIR}/{fn_base}.py", "w").write(code_str)
    try:
        if not re.search(r"def\s+build_weights\s*\(", code_str):
            raise SandboxError("truncated" if info["truncated"] else "no_code", "no build_weights definition in reply")
        W = validate(run_generated(code_str, timeout=SANDBOX_TIMEOUT["cifar" if cifar else "algo"]), shapes or TASKS[task]["shapes"])
        if cifar:
            m = load_weights(SmallCNN(), W); row.update(cifar_metrics(m, CIFAR_DATA[2], CIFAR_DATA[3])); row["ok"] = True
            row["_model"] = m
        else:
            T = TASKS[task]; m = load_weights(T["make_model"](), W); row.update(full_metrics(task, m, T)); row["ok"] = True
        np.savez_compressed(f"{GEN_DIR}/{fn_base}_weights.npz", **W)
    except SandboxError as e:
        row["error"], row["error_kind"] = str(e)[:500], e.kind
    except Exception as e:
        row["error"], row["error_kind"] = repr(e)[:500], classify_error(repr(e))
    if info["truncated"] and not row["ok"]: row["error_kind"] = "truncated"
    if cifar: row["success"] = None           # accuracy only for CIFAR
    else: row["success"] = bool(row["ok"] and row["all_acc"] >= SUCCESS_THRESH)
    return row

def synth_once(syn, task, level, rep, section="B_algo"):
    user = task_prompt(task, level) + (f"\n[[task={task}]]" if syn["kind"] == "mock" else "")
    tag = f"B_{task}_{level}_r{rep}"; t0 = time.perf_counter()
    base = {"task": task, "instance": "B", "synth": label_of(syn), "synth_requested": syn["model_id"], "synth_kind": syn["kind"],
            "prompt": level, "rep": rep}
    try:
        text, meta = call_llm(syn, SYSTEM_PROMPT, user, tag, rep=rep, section=section, max_tokens=CLAUDE_MAX_TOKENS["algo"])
    except BudgetExceeded: raise
    except Exception as e:
        return {**base, "ok": False, "all_acc": 0.0, "success": False, "error": repr(e)[:400], "error_kind": "llm_call", "synth_wall_s": time.perf_counter() - t0}
    row = {**base, **{k: v for k, v in meta.items()}, **process_reply(task, text, meta, f"{re.sub('[^A-Za-z0-9_.-]', '_', label_of(syn))}_{task}_{level}_r{rep}")}
    row["dry_run"] = meta.get("reply_source") in ("dry_run_mock",)   # dry-run rows are excluded from all tables
    row["synth_wall_s"] = time.perf_counter() - t0
    return row

def run_B_local(syn):
    reps = 1 if LOCAL_DETERMINISTIC else N_REPEATS_LOCAL
    deadline = time.time() + TIME_LIMIT["B_hf_each"]
    for rep in range(reps):
        for task in TASKS:
            for level in PROMPT_LEVELS:
                if time.time() > deadline: print(f"[{label_of(syn)}] time limit hit"); return "timeout"
                row = synth_once(syn, task, level, rep); PER_RUN.append(row); save_all()
                print(f"[B {label_of(syn)} {task} {level} r{rep}] ok={row['ok']} acc={row['all_acc']:.4f} {row.get('error_kind') or ''}")
    return "done"

## CIFAR-10: data, A baselines, B helper

In [ ]:
CIFAR_SHAPES = {"conv1.weight": (32, 3, 5, 5), "conv1.bias": (32,), "conv2.weight": (64, 32, 3, 3), "conv2.bias": (64,),
                "fc.weight": (10, 1024), "fc.bias": (10,)}
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 5, padding=2); self.conv2 = nn.Conv2d(32, 64, 3, padding=1); self.fc = nn.Linear(1024, 10)
    def features(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)          # 32x16x16
        x = F.adaptive_avg_pool2d(F.relu(self.conv2(x)), 4)  # 64x4x4
        return x.flatten(1)
    def forward(self, x): return self.fc(self.features(x))

CIFAR_CLASSES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
CIFAR_MEAN, CIFAR_STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)

def _cifar_parquet(root, split):
    """Fallback: the same CIFAR-10 images from the Hugging Face mirror (uoft-cs/cifar10), decoded with PIL."""
    import io, urllib.request
    from PIL import Image
    os.makedirs(root, exist_ok=True); f = os.path.join(root, f"{split}.parquet")
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"https://huggingface.co/datasets/uoft-cs/cifar10/resolve/main/plain_text/{split}-00000-of-00001.parquet", f)
    d = pd.read_parquet(f)
    x = np.stack([np.array(Image.open(io.BytesIO(v["bytes"])).convert("RGB")) for v in d["img"]])
    return torch.tensor(x).permute(0, 3, 1, 2).float() / 255., torch.tensor(d["label"].values.astype(np.int64))

def load_cifar():
    import torchvision, torchvision.transforms as tf
    root = "/kaggle/temp/cifar" if ON_KAGGLE else "./cifar_data"
    try:
        if os.environ.get("WSP_FAKE_CIFAR") == "1": raise RuntimeError("WSP_FAKE_CIFAR=1")
        tr = torchvision.datasets.CIFAR10(root, train=True, download=True); te = torchvision.datasets.CIFAR10(root, train=False, download=True)
        to = lambda d: (torch.tensor(d.data).permute(0, 3, 1, 2).float() / 255., torch.tensor(d.targets))
        (xtr, ytr), (xte, yte) = to(tr), to(te); src = "CIFAR-10"
        if QUICK: xtr, ytr, xte, yte = xtr[:3000], ytr[:3000], xte[:1000], yte[:1000]; src = "CIFAR-10-subset"
    except Exception as e0:
        print("torchvision CIFAR-10 failed:", repr(e0)[:160])
        try:
            if os.environ.get("WSP_FAKE_CIFAR") == "1": raise e0
            xtr, ytr = _cifar_parquet(root, "train"); xte, yte = _cifar_parquet(root, "test"); src = "CIFAR-10(hf-parquet)"
            if QUICK: xtr, ytr, xte, yte = xtr[:3000], ytr[:3000], xte[:1000], yte[:1000]; src += "-subset"
            m = torch.tensor(CIFAR_MEAN)[None, :, None, None]; s = torch.tensor(CIFAR_STD)[None, :, None, None]
            return (xtr - m) / s, ytr, (xte - m) / s, yte, src
        except Exception as e:
            print("CIFAR-10 unavailable -> synthetic stand-in (pipeline test only):", repr(e)[:120])
            g = torch.Generator().manual_seed(0); xtr = torch.rand(2000, 3, 32, 32, generator=g); ytr = torch.randint(0, 10, (2000,), generator=g)
            xte = torch.rand(500, 3, 32, 32, generator=g); yte = torch.randint(0, 10, (500,), generator=g); src = "FAKE"
    m = torch.tensor(CIFAR_MEAN)[None, :, None, None]; s = torch.tensor(CIFAR_STD)[None, :, None, None]
    return (xtr - m) / s, ytr, (xte - m) / s, yte, src

CIFAR_PROMPT = f"""Write the weights of a small CNN for CIFAR-10 classification WITHOUT any training, data, or gradients.
Input: 3x32x32 RGB images normalized per channel with mean {CIFAR_MEAN} and std {CIFAR_STD}. Classes (index order): {CIFAR_CLASSES}.
Architecture (PyTorch, fixed): conv1 = Conv2d(3,32,5,padding=2) -> ReLU -> MaxPool2d(2) -> conv2 = Conv2d(32,64,3,padding=1) -> ReLU
-> AdaptiveAvgPool2d(4) -> flatten (1024 = 64 channels x 4 x 4, channel-major) -> fc = Linear(1024,10).
Design meaningful filters from first principles (e.g. oriented edge/Gabor, color-opponent, center-surround, blob detectors in conv1;
combinations/texture detectors in conv2) and a classifier head encoding your prior knowledge of which colors, textures and spatial
layouts indicate each class (e.g. blue/uniform top regions for airplane/ship, green/brown textures for frog/deer, etc.).
Parameter names and shapes: {json.dumps({k: list(v) for k, v in CIFAR_SHAPES.items()})}
{FORMAT_RULES}"""

@torch.no_grad()
def eval_cifar(m, X, Y, noise=0.0, bs=1000):
    m.eval().to(DEVICE); c = 0; g = torch.Generator().manual_seed(0)
    for i in range(0, len(Y), bs):
        x = X[i:i+bs]
        if noise > 0: x = x + noise * torch.randn(x.shape, generator=g)
        lg = torch.nan_to_num(m(x.to(DEVICE)).float()); c += (lg.argmax(-1).cpu() == Y[i:i+bs]).sum().item()
    return c / len(Y)

def cifar_metrics(m, Xte, Yte):
    r = {"all_acc": eval_cifar(m, Xte, Yte)}
    r["test_acc"] = r["all_acc"]; r["above_chance_pts"] = r["all_acc"] - CIFAR_CHANCE
    for nz in NOISES: r[f"acc_noise{nz}"] = eval_cifar(m, Xte, Yte, nz)
    r.update(structure("cifar", get_weights(m))); return r

def train_A_cifar(Xtr, Ytr, Xval, Yval, opt_name, lr, seed, deadline):
    set_seed(seed); m = SmallCNN().to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=lr) if opt_name == "adam" else torch.optim.SGD(m.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    g = torch.Generator().manual_seed(seed); comp = 0.0; rows = []
    for step in range(1, max(CIFAR_STEPS) + 1):
        t0 = time.perf_counter(); idx = torch.randint(0, len(Ytr), (CIFAR_BATCH,), generator=g)
        x, y = Xtr[idx].to(DEVICE), Ytr[idx].to(DEVICE)
        if torch.rand(1, generator=g).item() < 0.5: x = x.flip(3)
        m.train(); loss = F.cross_entropy(m(x), y); opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        if DEVICE == "cuda": torch.cuda.synchronize()
        comp += time.perf_counter() - t0
        if step in CIFAR_STEPS:
            rows.append({"task": "cifar", "opt": opt_name, "lr": lr, "seed": seed, "step": step, "train_loss": loss.item(),
                         "val_acc": eval_cifar(m, Xval, Yval), "train_compute_s": comp})
        if time.time() > deadline: break
    return m, rows

def ridge_head(m, Xtr, Ytr, lam=1.0, n=10000):
    """Closed-form ridge readout on frozen (LLM-written) conv features. Uses data, no gradients."""
    with torch.no_grad():
        Fs = torch.cat([m.features(Xtr[i:i+1000].to(DEVICE)).cpu() for i in range(0, min(n, len(Ytr)), 1000)]).double()
        Y1 = F.one_hot(Ytr[:len(Fs)], 10).double() * 2 - 1
        mu, sd = Fs.mean(0), Fs.std(0) + 1e-6; Z = torch.cat([(Fs - mu) / sd, torch.ones(len(Fs), 1, dtype=torch.float64)], 1)
        Wr = torch.linalg.solve(Z.T @ Z + lam * len(Fs) * torch.eye(Z.shape[1], dtype=torch.float64), Z.T @ Y1)   # (1025,10)
        Wf = (Wr[:-1] / sd[:, None]).T; bf = Wr[-1] - (mu / sd) @ Wr[:-1]
        m2 = SmallCNN(); m2.load_state_dict(m.state_dict())
        m2.fc.weight.copy_(Wf.float()); m2.fc.bias.copy_(bf.float())
    return m2.to(DEVICE)

CIFAR_DATA = None
def run_cifar_A():
    global CIFAR_DATA
    Xtr, Ytr, Xte, Yte, src = CIFAR_DATA
    nval = min(5000, len(Ytr) // 10); Xval, Yval, Xt, Yt = Xtr[-nval:], Ytr[-nval:], Xtr[:-nval], Ytr[:-nval]
    deadline = time.time() + TIME_LIMIT["A_cifar"]
    for opt_name, lrs in CIFAR_LR_GRID.items():
        for lr in lrs:
            for seed in SEEDS_A:
                if time.time() > deadline: print("A_cifar time limit"); return
                try:
                    m, rows = train_A_cifar(Xt, Yt, Xval, Yval, opt_name, lr, seed, deadline); A_CURVES.extend(rows); last = rows[-1]
                    r = cifar_metrics(m, Xte, Yte)
                    PER_RUN.append({"task": "cifar", "instance": "A", "synth": opt_name, "prompt": "-", "rep": seed, "lr": lr, "steps": last["step"],
                                    "val_acc": last["val_acc"], "compute_s": last["train_compute_s"], "ok": True, "data": src, **r})
                    print(f"[A cifar {opt_name} lr={lr} s={seed}] val {last['val_acc']:.3f} test {r['test_acc']:.3f}")
                except Exception as e: print("A cifar run failed", repr(e))
                save_all()

if RUN_TASKS.get("cifar"):
    try:
        CIFAR_DATA = load_cifar(); print("CIFAR source:", CIFAR_DATA[4], CIFAR_DATA[0].shape)
        if RUN_A: run_cifar_A()
    except Exception as e: print("CIFAR A section failed:", repr(e)); traceback.print_exc()

def cifar_once(syn, rep, section="B_cifar"):
    Xtr, Ytr, Xte, Yte, src = CIFAR_DATA; t0 = time.perf_counter()
    base = {"task": "cifar", "instance": "B", "synth": label_of(syn), "synth_requested": syn["model_id"], "synth_kind": syn["kind"],
            "prompt": "full", "rep": rep, "data": src}
    try:
        text, meta = call_llm(syn, SYSTEM_PROMPT, CIFAR_PROMPT, f"B_cifar_r{rep}", rep=rep, section=section, max_tokens=CLAUDE_MAX_TOKENS["cifar"])
    except BudgetExceeded: raise
    except Exception as e:
        PER_RUN.append({**base, "ok": False, "all_acc": 0.0, "error": repr(e)[:400], "error_kind": "llm_call"}); return
    base["dry_run"] = meta.get("reply_source") in ("dry_run_mock",)
    row = {**base, **meta, **process_reply("cifar", text, meta, f"{re.sub('[^A-Za-z0-9_.-]', '_', label_of(syn))}_cifar_r{rep}", CIFAR_SHAPES, cifar=True)}
    m = row.pop("_model", None)
    if m is not None:
        try:
            rr = cifar_metrics(ridge_head(m, Xtr, Ytr), Xte, Yte)
            PER_RUN.append({**row, "instance": "B+ridge", **rr, "synth_wall_s": time.perf_counter() - t0})
        except Exception as e: print("ridge failed", repr(e))
    row["synth_wall_s"] = time.perf_counter() - t0; PER_RUN.append(row); save_all()
    print(f"[B cifar {label_of(syn)} r{rep}] ok={row['ok']} acc={row['all_acc']:.4f} {row.get('error_kind') or ''}")

## Local synthesizers

In [ ]:
# ---------------- Local open-weight synthesizers (no API cost). One model loaded at a time. ----------------
SYN_STATUS = {}
if RUN_LOCAL:
    for syn in SYNTHESIZERS:
        status = []
        try:
            if syn["kind"] == "hf": load_hf(syn)
            if TASKS: status.append("algo:" + run_B_local(syn))
            if CIFAR_DATA is not None:
                for rep in range(1 if LOCAL_DETERMINISTIC else N_REPEATS_CIFAR_LOCAL): cifar_once(syn, rep)
                status.append("cifar:done")
        except Exception as e: status.append("FAILED/SKIPPED " + repr(e)[:200]); print(f"synthesizer {syn['model_id']}:", repr(e)[:300])
        finally:
            if syn["kind"] == "hf": unload_hf()
        SYN_STATUS[syn["model_id"]] = {"label_in_tables": label_of(syn) if syn.get("resolved_id") or syn["kind"] != "hf" else None,
                                       "status": status}
        save_all()
print(json.dumps(SYN_STATUS, indent=1))

## Claude (capped, cached, dry-run by default)

In [ ]:
# ---------------- Claude: capped re-run of selected cases only ----------------
CLAUDE_SYN = {"name": "claude", "kind": "claude", "model_id": CLAUDE_MODEL_ID}
def claude_plan():
    """Per-case plan with cache status and worst-case cost (cached cases cost $0)."""
    rows = []
    for task, level, reps in RUN_CLAUDE_ONLY_CASES:
        if task != "cifar" and task not in TASKS: continue
        user = CIFAR_PROMPT if task == "cifar" else task_prompt(task, level)
        mt = CLAUDE_MAX_TOKENS["cifar" if task == "cifar" else "algo"]
        for rep in reps:
            hit = cache_get(CLAUDE_MODEL_ID, SYSTEM_PROMPT, user, rep, mt) is not None
            rows.append({"task": task, "prompt": level, "rep": rep, "cached": hit, "section": "B_cifar" if task == "cifar" else "B_algo",
                         "worst_usd": 0.0 if hit else worst_case_cost(user, SYSTEM_PROMPT, mt)})
    return pd.DataFrame(rows)

PLAN = claude_plan()
if len(PLAN):
    print(PLAN.groupby(["task", "prompt", "section"]).agg(n=("rep", "size"), cached=("cached", "sum"), worst_usd=("worst_usd", "sum")))
    print("PRE-RUN ESTIMATE: worst-case total $%.2f | caps: total $%.2f, sections %s | DRY_RUN_CLAUDE=%s RUN_CLAUDE=%s"
          % (PLAN.worst_usd.sum(), CLAUDE_BUDGET_USD, CLAUDE_SECTION_CAP, DRY_RUN_CLAUDE, RUN_CLAUDE))
if RUN_CLAUDE:
    st = []
    try:
        for _, p in PLAN.iterrows():
            if p.task == "cifar":
                if CIFAR_DATA is not None: cifar_once(CLAUDE_SYN, int(p.rep))
            else:
                row = synth_once(CLAUDE_SYN, p.task, p.prompt, int(p.rep))
                PER_RUN.append(row); save_all()
                print(f"[B claude {p.task} {p.prompt} r{p.rep}] src={row.get('reply_source')} ok={row['ok']} acc={row['all_acc']:.4f} {row.get('error_kind') or ''}")
        st.append("done")
    except BudgetExceeded as e: st.append("BUDGET_STOP"); print("BUDGET STOP:", e)
    SYN_STATUS["claude"] = {"status": st, "spent_usd": BUDGET.spent, "by_section": BUDGET.by_section, "api_calls": API_CALLS["claude"]}
    save_all()
print("Claude: real API calls %d | spent $%.4f" % (API_CALLS["claude"], BUDGET.spent))

## Instance C (Claude editing; off by default)

In [ ]:
def error_report(task, model, T):
    r = evaluate(model, T["X"], T["Y"]); W = get_weights(model); wrong = r["wrong"]
    X, Y = T["X"], T["Y"]
    rep = {"acc": r["acc"], "n_wrong": int((r["pred"] != Y).sum()), "min_margin_norm": r["min_margin_norm"], "min_prob_margin": r["min_prob_margin"],
           "dead_hidden_units": int((np.abs(W["W1"]).sum(1) < 1e-8).sum()),
           "param_norms": {k: float(np.linalg.norm(v)) for k, v in W.items()}}
    if task in ("modadd", "modmul", "permadd"):
        ab = [(int(X[i, :P].argmax()), int(X[i, P:].argmax()), int(Y[i]), int(r["pred"][i])) for i in wrong[:20]]
        rep["sample_wrong (a,b,true,pred)"] = ab; rep["fourier_top10_energy"] = fourier_energy(task, W)
    else:
        rep["sample_wrong (bits,true,pred)"] = [("".join(str(int(v)) for v in X[i].tolist()), int(Y[i]), int(r["pred"][i])) for i in wrong[:20]]
    return rep

def damage_weights(W, seed=0):
    g = np.random.default_rng(seed); W = {k: v.copy() for k, v in W.items()}; H = W["W1"].shape[0]
    dead = g.choice(H, int(0.8 * H), replace=False); W["W1"][dead] = 0; W["W2"][:, dead] = 0
    if "b1" in W: W["b1"][dead] = 0
    W["W1"] += 0.5 * g.standard_normal(W["W1"].shape); return W

C_PROMPT = """You are repairing the weights of the network below WITHOUT gradients or training.
{spec}
The current weights are available to your code as a dict W of numpy arrays (same names/shapes as above).
Error report of the current weights (JSON): {report}
Return ONLY one python code block defining `def edit_weights(W):` that returns a dict with ALL parameters (same names and shapes).
Use only numpy (np) and math. You may rewrite any part of the weights."""

def run_C():
    claude = CLAUDE_SYN
    deadline = time.time() + TIME_LIMIT["C"]
    for task, T in TASKS.items():
        fails = [r for r in PER_RUN if r.get("instance") == "B" and r["task"] == task and r["synth"] == label_of(claude)
                 and r["ok"] and not r["success"]][:C_MAX_TARGETS_PER_TASK]
        targets = [("damaged_oracle", damage_weights(ORACLE_W[task]))]
        for r in fails:
            f = f"{GEN_DIR}/{label_of(claude)}_{task}_{r['prompt']}_r{r['rep']}_weights.npz"
            if os.path.exists(f): targets.append((f"B_{r['prompt']}_r{r['rep']}", dict(np.load(f))))
        for name, W0 in targets:
            if time.time() > deadline: print("C time limit"); return
            cur = W0; m = load_weights(T["make_model"](), cur); acc0 = evaluate(m, T["X"], T["Y"])["acc"]; best = (acc0, cur)
            row = {"task": task, "target": name, "acc_before": acc0, "iters": [], "dry_run": DRY_RUN_CLAUDE}
            for it in range(C_ITERS):
                try:
                    rep = error_report(task, m, T)
                    text, meta = call_llm(claude, SYSTEM_PROMPT, C_PROMPT.format(spec=task_prompt(task, "full"), report=json.dumps(rep, default=str)),
                                          f"C_{task}_{name}_it{it}", rep=it, section="C", max_tokens=CLAUDE_MAX_TOKENS["C"])
                    code_str, _ = extract_code(text, "edit_weights", meta.get("stop_reason")); open(f"{GEN_DIR}/C_{task}_{name}_it{it}.py", "w").write(code_str)
                    cur = validate(run_generated(code_str, "edit_weights", inputs=cur), T["shapes"])
                    m = load_weights(T["make_model"](), cur); a = evaluate(m, T["X"], T["Y"])["acc"]
                    row["iters"].append({"it": it, "acc": a, **meta})
                    if a > best[0]: best = (a, cur)
                    if a >= 0.9999: break
                except BudgetExceeded as e: row["iters"].append({"it": it, "error": "BUDGET_STOP"}); break
                except Exception as e: row["iters"].append({"it": it, "error": repr(e)[:300]})
            row["acc_after"] = best[0]; C_ROWS.append(row); save_all()
            print(f"[C {task} {name}] {acc0:.4f} -> {best[0]:.4f}")
if RUN_C_CLAUDE and RUN_CLAUDE:
    try: run_C()
    except Exception as e: print("C section failed:", repr(e)); traceback.print_exc()

## Aggregation

In [ ]:
df = pd.DataFrame(PER_RUN).drop(columns=["pred"], errors="ignore")
if "dry_run" in df: df = df[df["dry_run"] != True]     # dry-run mock rows are never aggregated as results
# For A, keep only the LR chosen on validation (per task, optimizer), averaged over seeds.
sel = []
if len(df) and "lr" in df:
    for (t, s), g in df[df.instance == "A"].groupby(["task", "synth"]):
        best_lr = g.groupby("lr")["val_acc"].mean().idxmax(); sel.append(g[g.lr == best_lr])
dfA_sel = pd.concat(sel) if sel else pd.DataFrame()
dfagg_src = pd.concat([df[df.instance != "A"], dfA_sel]) if len(df) else df
agg = []
for keys, g in dfagg_src.groupby(["task", "instance", "synth", "prompt"], dropna=False):
    accs = g["all_acc"].fillna(0).tolist() if keys[1] != "A" else g["test_acc"].tolist()
    mu, sd, ci = t_ci(accs); n = len(g)
    is_cifar = keys[0] == "cifar"
    if is_cifar: k = None; lo = hi = np.nan
    else:
        k = int(g["success"].fillna(False).astype(bool).sum()) if keys[1] == "B" else int(sum(a >= SUCCESS_THRESH for a in accs)); lo, hi = wilson(k, n)
    row = {"task": keys[0], "instance": keys[1], "synth": keys[2], "prompt": keys[3], "n": n, "acc_mean": mu, "acc_std": sd, "acc_ci95": ci,
           "success_rate": (k / n if n else np.nan) if not is_cifar else np.nan,
           "note": (f"accuracy only; beats chance (10%) by >{CIFAR_ABOVE_CHANCE_PTS*100:.0f} pts: {mu - CIFAR_CHANCE > CIFAR_ABOVE_CHANCE_PTS}" if is_cifar else ""),
           "synth_requested": ",".join(sorted(set(map(str, g["synth_requested"].dropna())))) if "synth_requested" in g else "", "success_ci_lo": lo, "success_ci_hi": hi,
           "valid_code_rate": float(g["ok"].astype(bool).mean()) if "ok" in g else np.nan}
    if "error_kind" in g: row["error_kinds"] = json.dumps(g["error_kind"].dropna().value_counts().to_dict())
    if keys[1] == "A": row.update(lr=g["lr"].iloc[0], steps=g["steps"].mean(), compute_s=g["compute_s"].mean())
    for c in ["acc_noise0.1", "acc_noise0.3", "acc_noise0.5", "mean_margin_raw", "min_margin_raw", "mean_margin_norm", "min_margin_norm", "mean_prob_margin", "min_prob_margin", "W1_eff_rank", "W1_sparsity",
              "fourier_top10_energy", "in_tok", "out_tok", "cost_usd", "synth_wall_s"]:
        if c in g: row[c] = pd.to_numeric(g[c], errors="coerce").mean()
    agg.append(row)
dfagg = pd.DataFrame(agg); dfagg.to_csv(f"{OUT_DIR}/comparison_table.csv", index=False)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 250); dfagg

In [ ]:
# ---- Plots ----
try:
    dfc = pd.DataFrame(A_CURVES)
    tasks_all = sorted(set(dfagg.task)) if len(dfagg) else []
    if len(dfc):
        fig, axs = plt.subplots(1, len(tasks_all), figsize=(5 * len(tasks_all), 4), squeeze=False)
        for ax, t in zip(axs[0], tasks_all):
            g = dfc[dfc.task == t]
            for (o, lr), gg in g.groupby(["opt", "lr"]):
                mm = gg.groupby("step")[["val_acc", "train_compute_s"]].mean(); ax.plot(mm.index, mm.val_acc, "o-", ms=3, label=f"A {o} lr={lr}")
            for _, r in dfagg[(dfagg.task == t) & (dfagg.instance.isin(["B", "B+ridge"]))].iterrows():
                if r.prompt in ("full", "-") and np.isfinite(r.acc_mean): ax.axhline(r.acc_mean, ls=":", lw=1, label=f"{r.instance} {r.synth} {r.prompt}")
            ax.set_xscale("log"); ax.set_title(t); ax.set_xlabel("step"); ax.set_ylabel("accuracy"); ax.legend(fontsize=5)
        plt.tight_layout(); plt.savefig(f"{OUT_DIR}/accuracy_vs_steps.png", dpi=150); plt.show()
    b = dfagg[(dfagg.instance == "B") & (dfagg.task != "cifar")]
    if len(b):
        fig, axs = plt.subplots(1, len(set(b.task)), figsize=(5 * len(set(b.task)), 4), squeeze=False)
        for ax, (t, g) in zip(axs[0], b.groupby("task")):
            piv = g.pivot_table(index="prompt", columns="synth", values="success_rate")
            piv.plot.bar(ax=ax); ax.set_title(f"B success rate: {t}"); ax.set_ylim(0, 1.05)
        plt.tight_layout(); plt.savefig(f"{OUT_DIR}/B_success_by_prompt.png", dpi=150); plt.show()
    fig, axs = plt.subplots(1, max(1, len(tasks_all)), figsize=(5 * max(1, len(tasks_all)), 4), squeeze=False)
    for ax, t in zip(axs[0], tasks_all):
        for _, r in dfagg[dfagg.task == t].iterrows():
            if r.instance == "B" and r.prompt not in ("full",): continue
            ys = [r.acc_mean] + [r.get(f"acc_noise{n}", np.nan) for n in NOISES]; ax.plot([0] + NOISES, ys, "o-", ms=3, label=f"{r.instance} {r.synth}")
        ax.set_title(f"robustness: {t}"); ax.set_xlabel("input noise std"); ax.legend(fontsize=5)
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/robustness.png", dpi=150); plt.show()
except Exception as e: print("plotting failed:", repr(e)); traceback.print_exc()

In [ ]:
# ---- Final results.json + zip ----
results = {"config": {k: globals()[k] for k in ["QUICK", "MOCK", "RUN_TASKS", "PROMPT_LEVELS", "N_REPEATS_LOCAL",
           "N_REPEATS_CIFAR_LOCAL", "SUCCESS_THRESH", "CLAUDE_TEMPERATURE", "CLAUDE_MAX_TOKENS", "PRICE_PER_MTOK", "CLAUDE_BUDGET_USD",
           "CLAUDE_SECTION_CAP", "RUN_CLAUDE", "DRY_RUN_CLAUDE", "RUN_CLAUDE_ONLY_CASES", "CLAUDE_MODEL_ID",
           "LOCAL_MAX_NEW_TOKENS", "LOCAL_DETERMINISTIC", "LOCAL_TEMPERATURE", "LOCAL_TOP_P", "LOCAL_SEED_BASE", "SEEDS_A", "LR_GRID", "WEIGHT_DECAY", "CHECKPOINTS", "SPLIT",
           "CIFAR_STEPS", "CIFAR_LR_GRID", "P", "H_ALGO", "PAR_N", "PAR_K", "PAR_SUBSET", "G_ROOT"]}, "PERM": PERM.tolist(),
           "synthesizers": SYNTHESIZERS, "synth_status": SYN_STATUS, "llm_usage": USAGE, "claude_spend_usd": BUDGET.spent, "claude_spend_by_section": BUDGET.by_section, "claude_api_calls": API_CALLS["claude"],
           "env": {"torch": torch.__version__, "python": platform.python_version(), "device": DEVICE,
                   "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]},
           "elapsed_s": time.time() - T_START, "comparison": json.loads(dfagg.to_json(orient="records")),
           "per_run": json.loads(pd.DataFrame(PER_RUN).drop(columns=["pred"], errors="ignore").to_json(orient="records")), "C": C_ROWS}
json.dump(results, open(f"{OUT_DIR}/results.json", "w"), indent=1, default=str)
import shutil
_z = shutil.make_archive(os.path.join(tempfile.gettempdir(), "wsp_v3_outputs"), "zip", OUT_DIR)
zip_path = shutil.move(_z, os.path.join(OUT_DIR, "wsp_v3_outputs.zip"))
print("elapsed %.1f min | Claude $%.3f | zip: %s" % ((time.time() - T_START) / 60, BUDGET.spent, zip_path))